### 策略名称: 日内15分钟量价相关性因子 (Intraday 15-min PV Correlation)

**策略概述:**
本策略利用高频快照数据 (Snapshot)，将全天交易时间划分为多个 15 分钟的特定时间窗口。在每个窗口内，利用逐笔快照数据计算**价格收益率**与**成交量变化**之间的相关系数。

逻辑假设量价配合关系蕴含短期价格预测信息：
* **正相关 (Positive Correlation):** 价格上涨伴随放量，或下跌伴随缩量，通常代表趋势确认或买盘强势。
* **负相关 (Negative Correlation):** 价格上涨缩量，或下跌放量，通常代表背离或抛压沉重。
* **因子逻辑:** 捕捉该窗口内资金流向与价格变动的同步性，反映微观结构下的交易拥挤度或流动性供需。

---

**数学逻辑 (Mathematical Logic):**

1.  **中间价与快照差分 (Mid Price & Snapshot Delta):**
    $$P_{mid, t} = \frac{AskPrice1_t + BidPrice1_t}{2}$$
    $$r_t = \frac{P_{mid, t} - P_{mid, t-1}}{P_{mid, t-1} + 1e^{-8}}$$
    $$\Delta V_t = Volume_t - Volume_{t-1}$$
    * $r_t$: 逐笔快照间的收益率。
    * $\Delta V_t$: 逐笔快照间的增量成交量 (Snapshot Volume Delta)。

2.  **窗口聚合 (Window Aggregation):**
    对于每个 15 分钟窗口 $k$，收集窗口内的序列 $R_k = \{r_1, r_2, ..., r_n\}$ 和 $V_k = \{\Delta V_1, \Delta V_2, ..., \Delta V_n\}$。

3.  **因子构建 (Factor Construction):**
    $$Factor_k = \text{Corr}(R_k, V_k)$$
    * **Corr:** 计算皮尔逊相关系数 (Pearson Correlation)。
    * **处理:** 若窗口内有效样本数少于 3 个，则因子值设为 0，以避免统计不显著或计算错误。

---

**Args:**
* `datasource` (str): 数据源表名 (e.g., `'cpt_dwc_2026_stock_hs300_snapshot'`)
* `start_date` (str): 开始日期 `'YYYY-MM-DD HH:MM:SS'`
* `end_date` (str): 结束日期 `'YYYY-MM-DD HH:MM:SS'`

**Returns:**
* `pd.DataFrame`: 因子数据，包含 columns `['date', 'instrument', 'factor']`
    * 其中 `date` 为每个 15 分钟窗口的结束时间点。

In [ ]:
def main(datasource, start_date, end_date):
    """
    factor function
    构建 15分钟频率的量价相关性因子 (PV Correlation)

    Args:
        datasource (str): Datasource table name
        start_date (str): Start date in 'YYYY-MM-DD HH:MM:SS' format
        end_date (str): End date in 'YYYY-MM-DD HH:MM:SS' format

    Returns:
        pd.DataFrame: Factor data with columns ['date', 'instrument', 'factor']
    """
    import pandas as pd
    import dai

    sql = f"""
    -- 优化设置
    SET preserve_insertion_order=false;
    SET threads=4;

    WITH cte_snapshot AS (
        SELECT
            date,
            instrument_id,

            -- 用中间价替代 last_price，避免列不存在
            (ask_price1 + bid_price1) / 2 AS mid_price,

            -- 累计成交量
            volume,

            -- 交易日
            strftime(date, '%Y-%m-%d') AS trading_day,

            -- 按15分钟分段
            CASE
                -- 上午
                WHEN strftime(date, '%H%M') >= '0930' AND strftime(date, '%H%M') < '0945' THEN  94500
                WHEN strftime(date, '%H%M') >= '0945' AND strftime(date, '%H%M') < '1000' THEN 100000
                WHEN strftime(date, '%H%M') >= '1000' AND strftime(date, '%H%M') < '1015' THEN 101500
                WHEN strftime(date, '%H%M') >= '1015' AND strftime(date, '%H%M') < '1030' THEN 103000
                WHEN strftime(date, '%H%M') >= '1030' AND strftime(date, '%H%M') < '1045' THEN 104500
                WHEN strftime(date, '%H%M') >= '1045' AND strftime(date, '%H%M') < '1100' THEN 110000
                WHEN strftime(date, '%H%M') >= '1100' AND strftime(date, '%H%M') < '1115' THEN 111500
                WHEN strftime(date, '%H%M') >= '1115' AND strftime(date, '%H%M') <= '1130' THEN 113000

                -- 下午
                WHEN strftime(date, '%H%M') >= '1300' AND strftime(date, '%H%M') < '1315' THEN 131500
                WHEN strftime(date, '%H%M') >= '1315' AND strftime(date, '%H%M') < '1330' THEN 133000
                WHEN strftime(date, '%H%M') >= '1330' AND strftime(date, '%H%M') < '1345' THEN 134500
                WHEN strftime(date, '%H%M') >= '1345' AND strftime(date, '%H%M') < '1400' THEN 140000
                WHEN strftime(date, '%H%M') >= '1400' AND strftime(date, '%H%M') < '1415' THEN 141500
                WHEN strftime(date, '%H%M') >= '1415' AND strftime(date, '%H%M') < '1430' THEN 143000
                WHEN strftime(date, '%H%M') >= '1430' AND strftime(date, '%H%M') < '1445' THEN 144500
                WHEN strftime(date, '%H%M') >= '1445' AND strftime(date, '%H%M') < '1457' THEN 150000

                ELSE -1
            END AS time_segment
        FROM {datasource}
    ),

    cte_filtered AS (
        SELECT *
        FROM cte_snapshot
        WHERE time_segment != -1
          AND mid_price IS NOT NULL
          AND volume IS NOT NULL
    ),

    -- 先算窗口内逐笔差分序列
    cte_delta AS (
        SELECT
            trading_day,
            time_segment,
            instrument_id,
            date,

            mid_price,
            LAG(mid_price) OVER (
                PARTITION BY instrument_id, trading_day, time_segment
                ORDER BY date
            ) AS prev_mid_price,

            volume,
            LAG(volume) OVER (
                PARTITION BY instrument_id, trading_day, time_segment
                ORDER BY date
            ) AS prev_volume
        FROM cte_filtered
    ),

    -- 构造 ret/vol_delta 序列
    cte_series AS (
        SELECT
            trading_day,
            time_segment,
            instrument_id,

            (mid_price - prev_mid_price) / (prev_mid_price + 1e-8) AS ret,
            (volume - prev_volume) AS vol_delta
        FROM cte_delta
        WHERE prev_mid_price IS NOT NULL
          AND prev_volume IS NOT NULL
    ),

    -- 聚合成15分钟因子
    cte_window AS (
        SELECT
            trading_day,
            time_segment,
            instrument_id,

            -- 样本太少时给0，避免相关系数不稳定
            CASE
                WHEN COUNT(*) < 3 THEN 0
                ELSE COALESCE(CORR(ret, vol_delta), 0)
            END AS factor
        FROM cte_series
        GROUP BY instrument_id, trading_day, time_segment
    )

    SELECT
        CAST(CONCAT(
            f.trading_day,
            ' ',
            strftime(
                strptime(LPAD(CAST(f.time_segment AS VARCHAR), 6, '0'), '%H%M%S'),
                '%H:%M:%S'
            )
        ) AS DATETIME) AS date,
        all_instruments.instrument,
        f.factor
    FROM cte_window f
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(sql, filters={'date': [start_date, end_date]}).df()
    return df


if __name__ == '__main__':
    """
    开发调试专用模块：分块循环回测引擎
    (这部分代码保持不变，用于本地分段跑数据验证)
    """
    from bigmodule import M
    from datetime import datetime
    import pandas as pd
    import structlog
    import gc

    logger = structlog.get_logger()
    datasource = 'cpt_dwc_2026_stock_hs300_snapshot'

    # ==========================================
    # 配置回测时间范围
    # ==========================================
    full_start_date = '2023-04-01'
    full_end_date = '2023-04-01'

    date_ranges = pd.date_range(start=full_start_date, end=full_end_date, freq='MS')
    all_results = []

    logger.info(f"🚀 Starting PV Correlation Factor Backtest: {full_start_date} to {full_end_date}")

    for start_dt in date_ranges:
        current_start = start_dt.strftime('%Y-%m-%d 00:00:00')
        current_end = (start_dt + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d 23:59:59')

        logger.info(f"Processing Chunk: {current_start} => {current_end}")

        try:
            df_chunk = main(datasource, current_start, current_end)

            if df_chunk is not None and not df_chunk.empty:
                all_results.append(df_chunk)
                logger.info(f"✅ Chunk Done. Rows: {len(df_chunk)}")
            else:
                logger.warning(f"⚠️ Chunk Empty: {current_start}")

            del df_chunk
            gc.collect()

        except Exception as e:
            logger.error(f"❌ Error in chunk {current_start}: {e}")

    if all_results:
        logger.info("🧩 Concatenating all chunks...")
        final_data = pd.concat(all_results, ignore_index=True)
        final_data.sort_values(by=['date', 'instrument'], inplace=True)

        logger.info(f"🎉 All Done! Final Shape: {final_data.shape}")
        logger.info(f"Sample:\n{final_data.head()}")

        logger.info("📊 Starting Evaluation...")
        try:
            results = M.eval_dwc._latest(data=final_data)
        except Exception as e:
            logger.warning(f"Evaluation failed (local environment might miss specific modules): {e}")
            print("Data preview:", final_data.head())
    else:
        logger.error("No data generated.")
